# Create regression input dataset for mortality model

In [1]:
import xarray as xr
import xesmf as xe
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
import xagg as xa
import os
import re
import glob
import calendar
from matplotlib import pyplot as plt

from funcs_support import get_params,get_filepaths,utility_save
from funcs_aux import _remove_chunk_encoding
from funcs_preprocessing import rechunk_ds, regrid_ds, from_cache
dir_list = get_params()
from fsspec.implementations import local
fs = local.LocalFileSystem()

In [2]:
from distributed import Client

# Start dask client
client = Client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42685,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33823,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc2/proxy/45089/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:38445,


In [3]:
# Load STATA mortality panel dataset
df_dta = pd.read_stata(dir_list['aux']+'global_mortality_panel_public.dta')
years_insample = np.unique(df_dta.year.values)

# Load shapefile for insample regions
gdf = gpd.read_file(dir_list['aux']+'geo_data/mortality_insample_world.shp')
gdf['geometry'] = gdf['geometry'].make_valid()
# Avoids odd error in xa.pixel_overlaps
gdf = gdf.to_crs('EPSG:4326')

# Turn EU ISOs into "EU"
df_dta = df_dta.rename({'iso':'iso0'},axis=1)
euro_isos = [
  "AUT","BEL","BGR","CHE","CYP","CZE","DEU","DNK","ESP",
  "EST","FIN","GBR","GRC","HRV","HUN","IRL","ISL","ITA",
  "LIE","LTU","LUX","LVA","MKD","MLT","MNE","NLD","NOR",
  "POL","PRT","ROU","SVK","SVN","SWE","TUR"] 
# note that this does not include France, but includes Turkey; 
# this is because French data was gathered from a different dataset
# (whereas Turkish data is included in the rest of Europe)
df_dta['iso'] = df_dta.iso0.where([i not in euro_isos for i in df_dta.iso0],'EU')

# Issue with none-type ADM2 ids in Japan (prep_data_* does the same)
df_dta.loc[df_dta.iso == "JPN",'adm2_id'] = df_dta.loc[df_dta.iso == "JPN",'adm1_id']
gdf.loc[gdf.iso == "JPN",'adm2_id'] = gdf.loc[gdf.iso == "JPN",'adm1_id']


# Add in adm0 codes and subset to public insample regions
gdf_sub = pd.merge(df_dta.loc[:,['iso','adm0','adm1_id','adm2_id']].drop_duplicates(),
         gdf,
         on=['iso','adm1_id','adm2_id'])
# Make geodataframe again
gdf_sub = {'all':gdf_sub.set_geometry(gdf_sub['geometry'])}

# Make ADM1 and ADM0 level versions of the polygons
gdf_sub['adm1'] = gdf_sub['all'].dissolve(by=['iso','adm1_id']).reset_index().drop('adm2_id',axis=1)
gdf_sub['adm0'] = gdf_sub['all'].dissolve('adm0').reset_index().drop(['adm1_id','adm2_id'],axis=1)

# Subset reference dataset to index and socioeconomic variables
df_dta_out = (df_dta.set_index(['iso','adm0','adm1_id','adm2_id','year','agegroup']).
 sort_index().
 loc[:,['adm1','adm2','deaths','population','deathrate',
        *[c for c in df_dta.columns if (('loggdp' in c) and ('13br' not in c))]]])
df_dta_out = df_dta_out.copy()

In [4]:
gdf_sub['all'].to_file(dir_list['aux']+'geo_data/mortality_insample_world_wadm0.shp')

In [24]:
pop_fn = dir_list['aux']+'pop_data/gpw_v4_population_count_rev11_025deg_rg.nc'
if not os.path.exists(pop_fn):
    # Load population density
    pop = xr.open_dataset(dir_list['aux']+'pop_data/gpw_v4_population_count_rev11_2pt5_min.nc')
    pop = xa.fix_ds(pop)
    pop = pop.isel(raster=0) # for 2000 population
    
    var = [v for v in pop][0]
    pop = pop.rename({var:'pop'})
    # Otherwise divide by 0 error in xa.aggregate, should open a github issue
    pop['pop'] = pop['pop'].where(pop['pop']>0,0.0000000000001)
    
    params_subset = {'lat':slice(-56,86),'lon':slice(-180,180)}
    fine_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.125,
                                                                 params_subset['lat'].stop-0.124,0.25)),
                                        'lon':(['lon'],np.arange(params_subset['lon'].start+0.125,
                                                                 params_subset['lon'].stop-0.124,0.25))})
    
    # Regrid
    rgrd = xe.Regridder(pop,fine_grid,'conservative')
    pop_rg = rgrd(pop)
    
    # Save
    pop_rg.attrs['SOURCE'] = '1a_create_reginputs_mortality.ipynb'
    pop_rg.attrs['DESCRIPTION'] = '2.5-minute 2000 GPWv4 population, regridded conservatively to reference 0.25-degree grid, with 0s replaced by 1e-13 for `xagg` processing.'
    
    utility_save(pop_rg,pop_fn)

pop = xr.open_dataset(pop_fn)

In [27]:
mods = ['ERA5-025','MERRA2','JRA-3Q','GMFD','ERA5-025-1deg','MERRA2-1deg','JRA-3Q-1deg','GMFD-1deg']
regrid = True # only for non-1deg models
overwrite = True

varlist = ['tas','pr'] # This isn't fully dynamic. Lots of hardcoded things

# Whether to use a variable called "prcorr" isntead of precip for a given model
# (MERRA2 has a bias-corrected precip dataset that is saved here under prcorr)
prcorr_mods = ['MERRA2','MERRA2-1deg']

# What years to use for long-range T
lr_bnds = [1981,2010]

for mod in mods:
    print(f'\nProcessing {mod}')
    final_fn = dir_list['proc']+'reg_inputs/mortality/reginputs_mortality_'+mod+'_regvars_fulllength.csv'

    if overwrite or not os.path.exists(final_fn):
        # Get model tas, and pr
        df = get_filepaths()
        if mod in prcorr_mods:
            df = df.query('model == "'+mod+'" and (varname == "tas" or varname == "prcorr") and freq == "day"')
        else:
            df = df.query('model == "'+mod+'" and (varname == "tas" or varname == "pr") and freq == "day"')
        # Captures "global" (no subset suffix) files in this filesystem
        df = df.query('suffix != suffix')
        
        #------------- Prep data -------------
        fns_out = {'rchk':dir_list['proc']+'tmp/tmp_rchk.zarr',
                   'rgrd':dir_list['proc']+'tmp/tmp_rgrd.zarr',
                   'rgrd_rchk':dir_list['proc']+'tmp/tmp_rgrd_rchk.zarr'}
        
        ds = xr.open_mfdataset(df.path.values,join='inner')
        if mod in prcorr_mods:
            # "Bias-corrected" precip renamed to standard precip
            ds = ds.rename({'prcorr':'pr'})
        
        # Subset to just needed years
        ds = ds.isel(time = [t.dt.year.values in years_insample for t in ds.time])
        
        output_fns = {var:(dir_list['proc']+mod+'/'+var+'poly_ann_'+mod+'_historical_reanalysis_'+
                             str(ds.time.dt.year.min().values)+'0101-'+str(ds.time.dt.year.max().values)+'1231.zarr')
                      for var in varlist}
    
        if (not fs.exists(output_fns['tas']+'/.done')) or (not fs.exists(output_fns['pr']+'/.done')):
            if '1deg' in mod:
                # For 1-degree already regridded data, only need to rechunk
                if not fs.exists(fns_out['rgrd_rchk']+'/.done'):
                    # Rechunk to 1 chunk temporally
                    ds = ds.chunk({'time':-1,'lat':40,'lon':40}).unify_chunks()
                    
                    # Save rechunked as temporary zarr
                    utility_save(ds.drop_encoding(),fns_out['rgrd_rchk'],save_kwargs = {'zarr_format':3},zarr_mode='w')
            else:            
                #-------- Rechunk to one chunk for space --------
                if not fs.exists(fns_out['rchk']+'/.done'):
                    if mod == 'MERRA2':
                        # avoiding kernel panics, sigh
                        ds = ds.chunk({'lat':-1,'lon':-1,'time':10}).unify_chunks()
                    else:
                        # Rechunk to 1 chunk geographically
                        ds = ds.chunk({'lat':-1,'lon':-1,'time':20}).unify_chunks()
                    
                    # Save rechunked as temporary zarr
                    utility_save(ds.drop_encoding(),fns_out['rchk'],save_kwargs = {'zarr_format':3},zarr_mode='w')
                
                #-------- Regrid to 1x1 --------
                if not fs.exists(fns_out['rgrd']+'/.done'):
                    # Use decorator to manage loading / removing temporary file 
                    regrid_fromcache = from_cache(fns_out['rchk'],fns_out['rgrd'],fs,save_kwargs = {'zarr_format':3,'mode':'w'})(regrid_ds)
                    params_subset = {'lat':slice(-56,86),'lon':slice(-180,180)}
                    fine_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.125,
                        														 params_subset['lat'].stop-0.124,0.25)),
                        								'lon':(['lon'],np.arange(params_subset['lon'].start+0.125,
                        														 params_subset['lon'].stop-0.124,0.25))})
                    # Regrid to reference grid 
                    regrid_fromcache(ref_grid=fine_grid,regrid_method='bilinear')
                
                #-------- Rechunk to one chunk for time --------
                if not fs.exists(fns_out['rgrd_rchk']+'/.done'):
                    # Use decorator to manage loading / removing temporary file 
                    rechunk_fromcache = from_cache(fns_out['rgrd'],fns_out['rgrd_rchk'],fs,save_kwargs = {'zarr_format':3,'mode':'w'})(rechunk_ds)
                    # Rechunk
                    rechunk_fromcache({'time':-1,'lat':40,'lon':40})
            
            #------------- Calculate polynomials -------------
            ds = xr.open_zarr(fns_out['rgrd_rchk'])
    
            # Convert pr from kg/m^2/s to mm/day (which it looks like the original .dta is in)
            ds['pr'] = ds['pr']*60*60*24
            ds['pr'].attrs['units'] = 'mm / day'
            # Convert tas from K to C
            ds['tas'] = ds['tas']-273.15
            ds['tas'].attrs['units'] = 'deg C'
            
            # Calculate "polynomials of daily average temperatures [and precip] (up to fourth order),
            # summed across the year"
            ds_indv = xr.concat([ds**k for k in np.arange(1,5)],
                                dim=pd.Index(np.arange(1,5),name='degree'))
            ds_indv = ds_indv.resample(time='1YS').sum()
            
            # Get time as year instead
            ds_indv['time'] = ds_indv['time'].dt.year
            ds_indv = ds_indv.rename({'time':'year'})
            
            # Rechunk more coarsely in geography
            ds_indv = ds_indv.chunk({'degree':-1,'lat':40,'lon':40})
            
            for var in ds_indv:
                ds_out = ds_indv[[var]].rename({var:var+'poly'})
            
                ds_out[var+'poly'].attrs['long_name'] = 'sum of polynomial '+var
            
                ds_out.attrs['SOURCE'] = '1a_create_reginput_mortality.ipynb'
                ds_out.attrs['DESCRIPTION'] = 'sum of polynomials (i.e., sum_d X^k for d in 1:365, degree k)'
            
                output_fn = (dir_list['proc']+mod+'/'+var+'poly_ann_'+mod+'_historical_reanalysis_'+
                             str(ds_indv.year.min().values)+'0101-'+str(ds_indv.year.max().values)+'1231')
    
                if not fs.exists(output_fn+'.zarr'):
                
                    utility_save(ds_out.drop_encoding(),output_fn+'_tmp.zarr',save_kwargs = {'zarr_format':3})
                
                    # Now, load and rechunk in preparation for aggregation
                    ds_out = xr.open_dataset(output_fn+'_tmp.zarr')
                    ds_out = ds_out.chunk({'lat':-1,'lon':-1,'degree':1,'year':1})
                    utility_save(ds_out.drop_encoding(),output_fn+'.zarr',save_kwargs = {'zarr_format':3})
                    os.system('rm -rf '+output_fn+'_tmp.zarr')
                else:
                    print(output_fn+' exists, skipped.')
    
        else:
            print('All files already created for '+mod+', moving on to dataset creation!')
            
        # Open polynomial
        ds_indv = xr.open_mfdataset(dir_list['proc']+mod+'/*poly_ann_*'+
                                    str(ds.time.dt.year.min().values)+'0101-'+str(ds.time.dt.year.max().values)+'1231*')
    
        # Calculate long-run average temperature
        daysinyear = xr.DataArray([365+calendar.isleap(y) for y in ds_indv.year.values],
                                  dims = ('year'),
                                  coords = {'year':ds_indv.year.values})
        ds_indv['lr_tavg'] = (ds_indv.taspoly.sel(degree=1)/daysinyear).sel(year=slice(*lr_bnds)).mean('year')
        
        #------------- Aggregate -------------
        # Get weightmaps
        print('getting weightmaps...')
        with xa.set_options(silent=True):
            wms = {typ:xa.pixel_overlaps(ds_indv,geom,weights=pop.pop) 
                   for typ,geom in gdf_sub.items()}
    
        print('aggregating...')
        # Aggregate to polygons
        with xa.set_options(impl='numba',silent=True):
            aggs = {typ:xa.aggregate(ds_indv,wm) for typ,wm in wms.items()}
        
        
        #------------- Output -------------
        print('creating dataset...')
    
        idx_cols = ['iso','adm0','adm1_id','adm2_id','year']
        
        # Join in climate data
        # ACTUALLY WHAT WE WANT IS: 
        # adm2 taspoly, (prpoly really - verify using df_dta.query('agegroup=="total" and year == 2008').set_index(['adm0','adm1_id','adm2_id']).loc[:,'prcp_poly_1_GMFD'])
        # adm1 lr_tavg, prpoly (to compare with Alia's regs, she was using ADM1 precip)
    
        vars_bytyp = {'all':['taspoly','prpoly'],
                      'adm1':['lr_tavg']} #['prpoly','lr_tavg']
        renames = {'taspoly':'tavg_poly',
                   'prpoly':'prcp_poly'}
    
        dfs_out = []
        for typ in vars_bytyp:
            df_tmp = aggs[typ].to_dataframe().reset_index()
            df_tmp = df_tmp.loc[:,[col for col in df_tmp if col in [*idx_cols,'degree',*vars_bytyp[typ]]]]
            # Rename to fit STATA do file
            df_tmp = df_tmp.rename({var:rn for var,rn in renames.items() if var in vars_bytyp[typ]},axis=1)
            # Make degree a string variable
            df_tmp['degree'] = df_tmp['degree'].astype(str)
        
            non_poly_vals = [var for var in vars_bytyp[typ] if 'poly' not in var]
            if len(non_poly_vals)>0:
                non_poly_vals = df_tmp.set_index([col for col in df_tmp if col in idx_cols]).loc[:,*non_poly_vals]
        
            # Make wide in climvar x polydegree
            df_tmp = df_tmp.pivot(index=[col for col in df_tmp if col in idx_cols],
                columns = ['degree'],values = [col for col in df_tmp if '_poly' in col]) 
            df_tmp.columns = ['_'.join(c) for c in df_tmp.columns.to_flat_index()]
        
            # Put back in non polynomial values
            if len(non_poly_vals)>0:
                df_tmp = pd.merge(df_tmp,non_poly_vals,left_index=True,right_index=True)
        
            dfs_out.append(df_tmp)
        
        dfs_out = dfs_out[0].join(dfs_out[1]).drop_duplicates()
    
        # Add in socioeconomic variables
        dfs_out = dfs_out.join(df_dta_out)
        dfs_out = dfs_out.dropna()
    
        #---- Get in-sample mean T as well
        # Rename lr_tavg to have bounds in it
        dfs_out = dfs_out.rename({'lr_tavg':'lr_tavg_'+'_'.join([str(bnd) for bnd in lr_bnds])},
                       axis=1)
        
        # Get ADM1 aggregated values
        agg_tas_adm1 = aggs['adm1'].to_dataset()
        agg_tas_adm1 = agg_tas_adm1.set_coords(['adm1_id','adm0'])
        # Calculate mean T from taspoly1 by getting the average yearly T
        meanT = ((agg_tas_adm1['taspoly'].sel(degree=1) / 
         (365 + pd.to_datetime(agg_tas_adm1.year.astype(str)+'0101').is_leap_year)))
        
        # Get start / end years of data for each adm1
        lr_t_bnds = dfs_out.reset_index()[['year','adm0','adm1_id']].groupby(['adm0','adm1_id']).min().rename({'year':'yearmin'},axis=1).join(
        dfs_out.reset_index()[['year','adm0','adm1_id']].groupby(['adm0','adm1_id']).max().rename({'year':'yearmax'},axis=1)
        )
        
        # For each adm1, get average temperature over in-sample years
        lr_insamps = []
        for poly_idx in meanT.poly_idx.values:
            tas_tmp = meanT.sel(poly_idx=poly_idx).drop_vars('degree')
            df_idx = (str(tas_tmp.adm0.values),str(tas_tmp.adm1_id.values))
            if df_idx in lr_t_bnds.index:
                lr_insamps.append(tas_tmp.sel(year=slice(lr_t_bnds.loc[df_idx].yearmin,
                               lr_t_bnds.loc[df_idx].yearmax)).mean('year'))
        
        
        # Concatentate, put into df format
        meanT = xr.concat(lr_insamps,dim='poly_idx',coords='different',compat='equals')
        meanT = (meanT.to_dataframe().reset_index().drop('poly_idx',axis=1).
                 set_index(['adm0','adm1_id']).
                 rename({'taspoly':'lr_tavg_insamp'},axis=1))
        
        # Join into df
        dfs_out = dfs_out.join(meanT)

        #---- Save
        dfs_out.to_csv(final_fn)
        print(final_fn+' saved!')
    
        if fs.exists(fns_out['rgrd_rchk']):
            os.system('rm -rf '+fns_out['rgrd_rchk'])
    else:
        print(final_fn+' exists, skipped.')


Processing ERA5-025
All files already created for ERA5-025, moving on to dataset creation!
getting weightmaps...
aggregating...


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 116.74 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 116.74 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sendi

creating dataset...
/glade/derecho/scratch/schwarzwald/impact_uncertainty/reg_inputs/mortality/reginputs_mortality_ERA5-025_regvars_fulllength.csv saved!

Processing MERRA2
All files already created for MERRA2, moving on to dataset creation!
getting weightmaps...
aggregating...


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 116.72 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 116.72 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sendi

creating dataset...
/glade/derecho/scratch/schwarzwald/impact_uncertainty/reg_inputs/mortality/reginputs_mortality_MERRA2_regvars_fulllength.csv saved!

Processing JRA-3Q
All files already created for JRA-3Q, moving on to dataset creation!
getting weightmaps...
aggregating...


...

creating dataset...
/glade/derecho/scratch/schwarzwald/impact_uncertainty/reg_inputs/mortality/reginputs_mortality_MERRA2-1deg_regvars_fulllength.csv saved!

Processing JRA-3Q-1deg
All files already created for JRA-3Q-1deg, moving on to dataset creation!
getting weightmaps...
aggregating...


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 10.04 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 10.04 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending

creating dataset...
/glade/derecho/scratch/schwarzwald/impact_uncertainty/reg_inputs/mortality/reginputs_mortality_JRA-3Q-1deg_regvars_fulllength.csv saved!

Processing GMFD-1deg
All files already created for GMFD-1deg, moving on to dataset creation!
getting weightmaps...
aggregating...


/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 10.04 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 10.04 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/schwarzwald/.conda/envs/hle_iv/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending

creating dataset...
/glade/derecho/scratch/schwarzwald/impact_uncertainty/reg_inputs/mortality/reginputs_mortality_GMFD-1deg_regvars_fulllength.csv saved!


In [28]:
# Concatenate 0.25-deg data into single csv
fns = glob.glob(f'{dir_list['proc']}reg_inputs/mortality/*_regvars_*.csv')
fns = np.sort([fn for fn in fns if not re.search(r'\-1deg',fn)])
if len(fns)>0:
    dfs = [pd.read_csv(fn,low_memory=False) for fn in fns]
    
    for idx in range(len(dfs)):
        dfs[idx]['obs_mod'] = re.split(r'\_',re.split(r'\/',fns[idx])[-1])[2]
    
    dfs = pd.concat(dfs)
    
    dfs.to_csv(f'{dir_list['proc']}reg_inputs/mortality/reginputs_mortality_025deg.csv')
    
    # Delete tmp files
    for fn in fns:
        os.system('rm -rf '+fn)

In [29]:
# Concatenate 1-deg data into single csv
fns = np.sort(glob.glob(f'{dir_list['proc']}reg_inputs/mortality/*1deg_regvars_*.csv'))
if len(fns)>0:
    dfs = [pd.read_csv(fn,low_memory=False) for fn in fns]
    
    for idx in range(len(dfs)):
        dfs[idx]['obs_mod'] = re.sub(r'\-1deg','',re.split(r'\_',re.split(r'\/',fns[idx])[-1])[2])
    
    dfs = pd.concat(dfs)
    
    dfs.to_csv(f'{dir_list['proc']}reg_inputs/mortality/reginputs_mortality_1deg.csv')
    
    # Delete tmp files
    for fn in fns:
        os.system('rm -rf '+fn)